In [49]:
from sklearn.linear_model import LinearRegression,ElasticNet,Ridge,Lasso
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.metrics import r2_score,root_mean_squared_error
import pandas as pd
from sklearn.preprocessing import OneHotEncoder,StandardScaler

In [50]:
df=pd.read_csv(r'E:\DataAnalytics\Cases\Medical_Cost_Personal\insurance.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [51]:
X=df.drop('charges',axis=1)
y=df['charges']

In [52]:
ohc=OneHotEncoder(sparse_output=False,drop='first').set_output(transform='pandas') 
scaler=StandardScaler()
ct= make_column_transformer(
    (ohc, make_column_selector(dtype_include='object')),
    # (scaler,['age','bmi']),
            ("passthrough", make_column_selector(dtype_include='number')),
            verbose_feature_names_out=False)

ct=ct.set_output(transform='pandas')

X_ohe= ct.fit_transform(X)

X_ohe

,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest,age,bmi,children
0,0.0,1.0,0.0,0.0,1.0,19,27.900,0
1,1.0,0.0,0.0,1.0,0.0,18,33.770,1
2,1.0,0.0,0.0,1.0,0.0,28,33.000,3
3,1.0,0.0,1.0,0.0,0.0,33,22.705,0
4,1.0,0.0,1.0,0.0,0.0,32,28.880,0
...,...,...,...,...,...,...,...,...
1333,1.0,0.0,1.0,0.0,0.0,50,30.970,3
1334,0.0,0.0,0.0,0.0,0.0,18,31.920,0
1335,0.0,0.0,0.0,1.0,0.0,18,36.850,0
1336,0.0,0.0,0.0,0.0,1.0,21,25.800,0


In [53]:
X_train, X_test, y_train,y_test= train_test_split(X_ohe,y, random_state=26, test_size=0.3)

In [54]:
model=LinearRegression()
model.fit(X_train,y_train)

LinearRegression()

In [55]:
y_pred=model.predict(X_test)


In [56]:
print('R 2 score:-',r2_score(y_test,y_pred))
print('Root_Mean:-',root_mean_squared_error(y_test,y_pred))

R 2 score:- 0.7625873682287878
Root_Mean:- 6260.264421571414


In [57]:
# Ridge
import numpy as np
alpha=np.linspace(0.001,10,20)
scores=[]
for x in alpha:
    model=Ridge(alpha=x)
    model.fit(X_train,y_train)
    y_pred=model.predict(X_test)
    scores.append([x,r2_score(y_test,y_pred),root_mean_squared_error(y_test,y_pred)])
ridgeScr=pd.DataFrame(scores,columns=['Alpha','r2_Score','RMSE_Score']).sort_values('r2_Score',ascending=False)
ridgeScr.head()


,Alpha,r2_Score,RMSE_Score
0,0.001000,0.762587,6260.266579
1,0.527263,0.762494,6261.496241
2,1.053526,0.762387,6262.911404
3,1.579789,0.762265,6264.507807
4,2.106053,0.762131,6266.281271


In [59]:
# lasso
alpha=np.linspace(0.001,10,20)
scores=[]
for x in alpha:
    model=Lasso(alpha=x)
    model.fit(X_train,y_train)
    y_pred=model.predict(X_test)
    scores.append([x,r2_score(y_test,y_pred),root_mean_squared_error(y_test,y_pred)])
lassoScr=pd.DataFrame(scores,columns=['Alpha','r2_Score','RMSE_Score']).sort_values('r2_Score',ascending=False)
lassoScr.head()

,Alpha,r2_Score,RMSE_Score
0,0.001000,0.762587,6260.264476
1,0.527263,0.762585,6260.294981
2,1.053526,0.762583,6260.328481
3,1.579789,0.762580,6260.364813
4,2.106053,0.762577,6260.404481


In [ ]:
# ElasticNet
alpha=np.linspace(0.001,10,20)
l1_ratio=np.linspace(0.001,1,20)
scores=[]
for x in alpha:
    for y in l1_ratio:
        model=ElasticNet(alpha=x,l1_ratio=y)
        model.fit(X_train,y_train)
        y_pred=model.predict(X_test)
        scores.append([x,r2_score(y_test,y_pred),root_mean_squared_error(y_test,y_pred)])
elastScr=pd.DataFrame(scores,columns=['Alpha','r2_Score','RMSE_Score']).sort_values('r2_Score',ascending=False)
elastScr.head()

,Alpha,r2_Score,RMSE_Score
19,0.001000,0.762587,6260.264476
39,0.527263,0.762585,6260.294981
59,1.053526,0.762583,6260.328481
79,1.579789,0.762580,6260.364813
18,0.001000,0.762579,6260.371456


In [ ]:
pd.concat([ridgeScr.iloc[0,:2],lassoScr.iloc[0,0:2],elastScr.iloc[0,0:2]])

Alpha       0.001000
r2_Score    0.762587
Alpha       0.001000
r2_Score    0.762587
Alpha       0.001000
r2_Score    0.762587
dtype: float64